In [3]:
%pip install pandas numpy scikit-learn streamlit plotly openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

project_folder = Path(
    r"C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar"
)

processed_folder = project_folder / "data" / "processed"

print("Project folder:", project_folder.exists())
print("Processed folder:", processed_folder.exists())

Project folder: True
Processed folder: True


In [5]:
import pandas as pd
import joblib
from pathlib import Path

# Project paths
project_folder = Path(
    r"C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar"
)

processed_folder = project_folder / "data" / "processed"
models_folder = project_folder / "models"

# Load commodity data
csv_path = processed_folder / "commodity_monthly_changes.csv"

commodity_df = pd.read_csv(
    csv_path,
    parse_dates=["observation_date"]
)

# Load trained models
logistic_model = joblib.load(
    models_folder / "logistic_shock_model.pkl"
)

random_forest_model = joblib.load(
    models_folder / "random_forest_shock_model.pkl"
)

# Load selected thresholds
thresholds = joblib.load(
    models_folder / "model_thresholds.pkl"
)

# Display verification
print("DATA LOADED SUCCESSFULLY")
print("Dataset shape:", commodity_df.shape)
print("Commodities:", commodity_df["commodity_name"].unique())
print("Date range:",
      commodity_df["observation_date"].min(),
      "to",
      commodity_df["observation_date"].max())

print("\nModels loaded:")
print("Logistic Regression:", type(logistic_model).__name__)
print("Random Forest:", type(random_forest_model).__name__)
print("Thresholds:", thresholds)

DATA LOADED SUCCESSFULLY
Dataset shape: (4722, 5)
Commodities: ['Crude oil, Brent' 'Maize' 'Rice, Thai 5%' 'Urea' 'Wheat, US HRW' 'DAP']
Date range: 1960-01-01 00:00:00 to 2026-09-01 00:00:00

Models loaded:
Logistic Regression: Pipeline
Random Forest: Pipeline
Thresholds: {'logistic': 0.5, 'random_forest': 0.4}


In [6]:
import numpy as np
import pandas as pd

def prepare_latest_features(data):

    data = data.sort_values(
        ["commodity_name", "observation_date"]
    ).copy()

    # Current monthly price change
    data["monthly_change_pct"] = (
        data.groupby("commodity_name")["price"].pct_change() * 100
    )

    # Previous month's change
    data["previous_month_change_pct"] = (
        data.groupby("commodity_name")["monthly_change_pct"].shift(1)
    )

    # Average change over the latest three months
    data["three_month_avg_change_pct"] = (
        data.groupby("commodity_name")["monthly_change_pct"]
        .transform(lambda s: s.rolling(3, min_periods=3).mean())
    )

    latest_features = (
        data.groupby("commodity_name", group_keys=False)
        .tail(1)
        .dropna(subset=[
            "monthly_change_pct",
            "previous_month_change_pct",
            "three_month_avg_change_pct"
        ])
        .copy()
    )

    return latest_features


def generate_shock_predictions(data):

    latest_features = prepare_latest_features(data)

    numeric_features = [
        "monthly_change_pct",
        "previous_month_change_pct",
        "three_month_avg_change_pct"
    ]

    categorical_features = ["commodity_name"]

    model_features = latest_features[
        numeric_features + categorical_features
    ]

    # Generate probabilities
    logistic_probabilities = logistic_model.predict_proba(
        model_features
    )[:, 1]

    rf_probabilities = random_forest_model.predict_proba(
        model_features
    )[:, 1]

    latest_features["logistic_probability"] = logistic_probabilities
    latest_features["random_forest_probability"] = rf_probabilities

    # Apply selected thresholds
    latest_features["logistic_warning"] = (
        latest_features["logistic_probability"]
        >= thresholds["logistic"]
    )

    latest_features["random_forest_warning"] = (
        latest_features["random_forest_probability"]
        >= thresholds["random_forest"]
    )

    return latest_features


prediction_df = generate_shock_predictions(commodity_df)

display(
    prediction_df[[
        "commodity_name",
        "observation_date",
        "price",
        "monthly_change_pct",
        "three_month_avg_change_pct",
        "logistic_probability",
        "logistic_warning",
        "random_forest_probability",
        "random_forest_warning"
    ]].round(3)
)

,commodity_name,observation_date,price,monthly_change_pct,three_month_avg_change_pct,logistic_probability,logistic_warning,random_forest_probability,random_forest_warning
4716,"Crude oil, Brent",2026-09-01,116.8,28.493,11.715,0.773,True,0.808,True
4717,DAP,2026-09-01,800.6,0.895,0.712,0.511,True,0.287,False
4718,Maize,2026-09-01,239.6,6.964,6.734,0.503,True,0.491,True
4719,"Rice, Thai 5%",2026-09-01,482.0,2.335,-0.758,0.360,False,0.257,False
4720,Urea,2026-09-01,407.5,4.487,-3.244,0.611,True,0.415,True
4721,"Wheat, US HRW",2026-09-01,347.0,5.152,6.665,0.451,False,0.418,True


In [7]:
import json

shockradar_context = {
    "project": "4F ShockRadar AI",
    "data_source": "World Bank Commodity Markets",
    "latest_data_date": str(
        prediction_df["observation_date"].max().date()
    ),
    "prediction_target": (
        "A commodity price increase of 10% or more "
        "in the following month"
    ),
    "model_results": prediction_df[[
        "commodity_name",
        "price",
        "monthly_change_pct",
        "three_month_avg_change_pct",
        "logistic_probability",
        "logistic_warning",
        "random_forest_probability",
        "random_forest_warning"
    ]].round(3).to_dict(orient="records")
}

shockradar_json = json.dumps(
    shockradar_context,
    indent=2
)

print(shockradar_json)

{
  "project": "4F ShockRadar AI",
  "data_source": "World Bank Commodity Markets",
  "latest_data_date": "2026-09-01",
  "prediction_target": "A commodity price increase of 10% or more in the following month",
  "model_results": [
    {
      "commodity_name": "Crude oil, Brent",
      "price": 116.8,
      "monthly_change_pct": 28.493,
      "three_month_avg_change_pct": 11.715,
      "logistic_probability": 0.773,
      "logistic_warning": true,
      "random_forest_probability": 0.808,
      "random_forest_warning": true
    },
    {
      "commodity_name": "DAP",
      "price": 800.6,
      "monthly_change_pct": 0.895,
      "three_month_avg_change_pct": 0.712,
      "logistic_probability": 0.511,
      "logistic_warning": true,
      "random_forest_probability": 0.287,
      "random_forest_warning": false
    },
    {
      "commodity_name": "Maize",
      "price": 239.6,
      "monthly_change_pct": 6.964,
      "three_month_avg_change_pct": 6.734,
      "logistic_probability": 0

In [9]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:1b",
        "prompt": "Explain commodity price shocks in simple business language.",
        "stream": False
    },
    timeout=(10, 120)
)

print(response.json()["response"])

Imagine you're a farmer who grows wheat. You plant, nurture, and harvest your crop every year. But one year, a big storm hits, destroying a lot of your crops. This can cause a shortage of wheat, which drives up the price. That's kind of like what happens in the economy when there's a "commodity price shock."

A commodity price shock is when there's a big change in the price of a specific good or resource, like oil, food, or metals. This can be caused by various factors, such as:

* A sudden increase in demand (like a big wedding or a sudden need for energy)
* A supply shock (like a natural disaster that destroys a lot of crops or a plant that can't grow)
* A change in global demand (like a shift in consumer behavior or a new technology that affects production)

When a commodity price shock happens, it can have a ripple effect throughout the economy. It can:

* Increase production costs for businesses that use the commodity
* Reduce the profit margins for consumers who buy the commodity

In [10]:
import requests
import json

def generate_shockradar_report(prediction_df):

    context = prediction_df[[
        "commodity_name",
        "price",
        "monthly_change_pct",
        "three_month_avg_change_pct",
        "logistic_probability",
        "logistic_warning",
        "random_forest_probability",
        "random_forest_warning"
    ]].round(2).to_dict(orient="records")

    prompt = f"""
You are the AI analyst for 4F ShockRadar AI.

Analyse the following actual commodity data and ML predictions.

DATA:
{json.dumps(context, indent=2)}

Write a concise business risk report with these sections:

1. Executive Summary
2. Commodities flagged by both models
3. Commodities flagged by only one model
4. Commodities not flagged
5. Recommended monitoring actions
6. Important limitations

Rules:
- Use only the supplied data.
- Do not invent prices or statistics.
- Explain that probabilities are model estimates, not certainties.
- A warning means predicted probability meets the model's selected threshold.
- Do not claim that a price increase has already happened.
- The prediction target is a 10% or greater price increase
  in the following month.
- Use simple professional business language.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2,
                "num_predict": 500
            }
        },
        timeout=(10, 180)
    )

    response.raise_for_status()

    return response.json()["response"]


shockradar_report = generate_shockradar_report(prediction_df)

print(shockradar_report)

**Business Risk Report**

**Executive Summary**

This report analyzes the actual commodity data and ML predictions to identify potential business risks. The analysis reveals that certain commodities are flagged by both models as having a high probability of price increase in the following month. This report provides a concise summary of the findings, including commodities flagged by both models, commodities flagged by only one model, and commodities not flagged.

**Commodities Flagged by Both Models**

The following commodities are flagged by both the Logistic Probability and Random Forest Probability models:

* Crude oil, Brent
* Maize
* Rice, Thai 5%
* Urea

These commodities have a predicted probability of meeting the 10% or greater price increase target in the following month.

**Commodities Flagged by Only One Model**

The following commodities are flagged by only one model:

* DAP (Logistic Probability only)
* Wheat, US HRW (Random Forest Probability only)

**Commodities Not Flag

In [11]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
C:\ProgramData\anaconda3\python.exe

Python version:
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [12]:
import sys

!{sys.executable} -m pip install streamlit plotly pandas joblib scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [13]:
import streamlit
import plotly
import pandas
import joblib
import sklearn

print("All packages installed successfully!")
print("Streamlit:", streamlit.__version__)
print("Plotly:", plotly.__version__)

All packages installed successfully!
Streamlit: 1.51.0
Plotly: 6.3.0


In [14]:
from pathlib import Path

project_folder = Path(
    r"C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar"
)

print("Project folder exists:", project_folder.exists())

if project_folder.exists():
    print("\nFiles in project folder:")
    for item in project_folder.iterdir():
        print(item.name)

print("\nSearching for app.py...")
for path in Path(r"C:\Users\DELL\OneDrive\Desktop").rglob("app.py"):
    print(path)

Project folder exists: True

Files in project folder:
dashboard
data
models
notebooks
sql

Searching for app.py...
C:\Users\DELL\OneDrive\Desktop\my notebooks\.venv\Lib\site-packages\pythonwin\pywin\framework\app.py
C:\Users\DELL\OneDrive\Desktop\my notebooks\.venv\Lib\site-packages\prompt_toolkit\filters\app.py


In [15]:
from pathlib import Path

dashboard_folder = project_folder / "dashboard"

print("Dashboard folder exists:", dashboard_folder.exists())

if dashboard_folder.exists():
    print("\nDashboard folder contents:")
    for item in dashboard_folder.iterdir():
        print(item.name)

Dashboard folder exists: True

Dashboard folder contents:


In [16]:

from pathlib import Path

project_folder = Path(
    r"C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar"
)

dashboard_folder = project_folder / "dashboard"
dashboard_folder.mkdir(exist_ok=True)

app_code = r'''
import streamlit as st
import pandas as pd
import plotly.express as px
import joblib
import requests
from pathlib import Path

st.set_page_config(
    page_title="4F ShockRadar AI",
    page_icon="🌍",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_PATH = BASE_DIR / "data" / "processed" / "commodity_monthly_changes.csv"
MODEL_DIR = BASE_DIR / "models"

st.title("🌍 4F ShockRadar AI")
st.caption("AI-powered commodity price risk monitoring")

@st.cache_data
def load_data():
    data = pd.read_csv(DATA_PATH)
    data["observation_date"] = pd.to_datetime(data["observation_date"])
    return data

@st.cache_resource
def load_models():
    logistic = joblib.load(MODEL_DIR / "logistic_shock_model.pkl")
    forest = joblib.load(MODEL_DIR / "random_forest_shock_model.pkl")
    thresholds = joblib.load(MODEL_DIR / "model_thresholds.pkl")
    return logistic, forest, thresholds

try:
    df = load_data()
    logistic_model, rf_model, thresholds = load_models()
except Exception as error:
    st.error(f"Could not load project data or models: {error}")
    st.stop()

# Prepare latest features for each commodity
def get_predictions(data):
    data = data.sort_values(
        ["commodity_name", "observation_date"]
    ).copy()

    data["monthly_change_pct"] = (
        data.groupby("commodity_name")["price"].pct_change() * 100
    )

    data["previous_month_change_pct"] = (
        data.groupby("commodity_name")["monthly_change_pct"].shift(1)
    )

    data["three_month_avg_change_pct"] = (
        data.groupby("commodity_name")["monthly_change_pct"]
        .transform(lambda x: x.rolling(3, min_periods=3).mean())
    )

    latest = (
        data.groupby("commodity_name", group_keys=False)
        .tail(1)
        .dropna(subset=[
            "monthly_change_pct",
            "previous_month_change_pct",
            "three_month_avg_change_pct"
        ]).copy()
    )

    features = [
        "monthly_change_pct",
        "previous_month_change_pct",
        "three_month_avg_change_pct",
        "commodity_name"
    ]

    latest["logistic_probability"] = (
        logistic_model.predict_proba(latest[features])[:, 1]
    )

    latest["random_forest_probability"] = (
        rf_model.predict_proba(latest[features])[:, 1]
    )

    latest["logistic_warning"] = (
        latest["logistic_probability"] >= thresholds["logistic"]
    )

    latest["random_forest_warning"] = (
        latest["random_forest_probability"] >= thresholds["random_forest"]
    )

    return latest

predictions = get_predictions(df)

latest_date = df["observation_date"].max()

# Summary metrics
c1, c2, c3, c4 = st.columns(4)

c1.metric("Commodities tracked", df["commodity_name"].nunique())
c2.metric("Historical records", f"{len(df):,}")
c3.metric("Latest data", latest_date.strftime("%b %Y"))
c4.metric(
    "Flagged by both models",
    int((predictions["logistic_warning"] &
         predictions["random_forest_warning"]).sum())
)

st.divider()

# Navigation
page = st.sidebar.radio(
    "Dashboard sections",
    [
        "Overview",
        "Price Trends",
        "ML Risk Predictions",
        "AI Business Report"
    ]
)

if page == "Overview":
    st.subheader("Commodity Risk Overview")

    st.write(
        "Monitor historical international commodity prices and "
        "experimental machine learning warnings."
    )

    summary = predictions[[
        "commodity_name",
        "price",
        "monthly_change_pct",
        "logistic_probability",
        "random_forest_probability"
    ]].copy()

    summary["logistic_probability"] *= 100
    summary["random_forest_probability"] *= 100

    summary = summary.rename(columns={
        "commodity_name": "Commodity",
        "price": "Latest price",
        "monthly_change_pct": "Monthly change (%)",
        "logistic_probability": "Logistic probability (%)",
        "random_forest_probability": "Random Forest probability (%)"
    })

    st.dataframe(
        summary.round(2),
        use_container_width=True,
        hide_index=True
    )

    st.info(
        "The model estimates the probability of a price increase of "
        "10% or more in the following month. Warnings are not guarantees."
    )

elif page == "Price Trends":
    st.subheader("Historical Commodity Price Analysis")

    commodity = st.selectbox(
        "Choose a commodity",
        sorted(df["commodity_name"].unique())
    )

    selected = df[df["commodity_name"] == commodity].sort_values(
        "observation_date"
    )

    price_chart = px.line(
        selected,
        x="observation_date",
        y="price",
        title=f"{commodity} – Historical Price Trend"
    )

    st.plotly_chart(price_chart, use_container_width=True)

    change_chart = px.bar(
        selected,
        x="observation_date",
        y="monthly_change_pct",
        title="Monthly Percentage Change"
    )

    st.plotly_chart(change_chart, use_container_width=True)

elif page == "ML Risk Predictions":
    st.subheader("Machine Learning Risk Predictions")

    result = predictions.copy()

    def classify(row):
        if row["logistic_warning"] and row["random_forest_warning"]:
            return "Both models"
        if row["logistic_warning"]:
            return "Logistic only"
        if row["random_forest_warning"]:
            return "Random Forest only"
        return "Neither model"

    result["risk_category"] = result.apply(classify, axis=1)

    st.dataframe(
        result[[
            "commodity_name",
            "price",
            "monthly_change_pct",
            "logistic_probability",
            "random_forest_probability",
            "risk_category"
        ]].round(3),
        use_container_width=True,
        hide_index=True
    )

    chart_data = result.melt(
        id_vars="commodity_name",
        value_vars=[
            "logistic_probability",
            "random_forest_probability"
        ],
        var_name="Model",
        value_name="Probability"
    )

    chart_data["Probability"] *= 100

    fig = px.bar(
        chart_data,
        x="commodity_name",
        y="Probability",
        color="Model",
        barmode="group",
        title="Estimated Probability of Next-Month Price Shock",
        labels={"Probability": "Probability (%)"}
    )

    st.plotly_chart(fig, use_container_width=True)

    st.caption(
        "Thresholds: Logistic Regression = 0.50; "
        "Random Forest = 0.40. These are experimental model thresholds."
    )

elif page == "AI Business Report":
    st.subheader("AI-Generated Business Risk Report")

    st.write(
        "Generate a business summary using your local Ollama model. "
        "No OpenAI API key is required."
    )

    if st.button("Generate Business Report"):
        context = predictions[[
            "commodity_name",
            "price",
            "monthly_change_pct",
            "logistic_probability",
            "random_forest_probability",
            "logistic_warning",
            "random_forest_warning"
        ]].round(3).to_dict(orient="records")

        prompt = f"""
        You are a commodity risk analyst.
        Use only the following verified data: {context}

        Classify commodities correctly:
        - Both models: both warning values are True.
        - Logistic only: only logistic warning is True.
        - Random Forest only: only random forest warning is True.
        - Neither: both warning values are False.

        Write an executive summary, model findings, monitoring actions,
        and limitations. Do not invent figures.
        The target is a 10% or greater price increase in the following
        month. Probabilities are estimates, not guarantees.
        The latest data date is {latest_date.date()}.
        """

        try:
            with st.spinner("Generating report with local Llama model..."):
                response = requests.post(
                    "http://localhost:11434/api/generate",
                    json={
                        "model": "llama3.2:1b",
                        "prompt": prompt,
                        "stream": False,
                        "options": {
                            "temperature": 0.2,
                            "num_predict": 500
                        }
                    },
                    timeout=(10, 180)
                )

                response.raise_for_status()
                report = response.json()["response"]

            st.markdown(report)

        except requests.exceptions.RequestException as error:
            st.error(
                "Could not connect to Ollama. Make sure Ollama is "
                f"running on your computer. Details: {error}"
            )

st.divider()

st.caption(
    "Data source: World Bank Commodity Markets. "
    "Historical international prices, not Indian retail prices. "
    "Experimental decision-support system."
)
'''

app_path = dashboard_folder / "app.py"
app_path.write_text(app_code, encoding="utf-8")

print("Dashboard application created successfully!")
print("Location:", app_path)
print("File exists:", app_path.exists())


Dashboard application created successfully!
Location: C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar\dashboard\app.py
File exists: True


In [17]:

from pathlib import Path

project_folder = Path(
    r"C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar"
)

requirements = """streamlit
plotly
pandas
numpy
scikit-learn
joblib
requests
openpyxl
"""

gitignore = """# Python
__pycache__/
*.py[cod]
.venv/
venv/
.ipynb_checkpoints/

# Temporary files
~$*
*.log
.DS_Store

# Raw source data
data/raw/

# Streamlit
.streamlit/secrets.toml
"""

(project_folder / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

(project_folder / ".gitignore").write_text(
    gitignore, encoding="utf-8"
)

print("Created:")
print(project_folder / "requirements.txt")
print(project_folder / ".gitignore")


Created:
C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar\requirements.txt
C:\Users\DELL\OneDrive\Desktop\projects\4F Shock radar\.gitignore
